# 03 — Gold Enrichment (fixed)

This notebook is adapted to work with the fixed Silver notebook and your current
Databricks + Unity Catalog + Serverless setup.

It keeps the key logic from the tutorial's Gold notebook:

- read Silver Parquet data
- enrich coordinates with `country_code`
- classify earthquake significance as Low / Moderate / High
- write analytics-ready Gold Parquet data

It writes one date partition at a time so rerunning the same date does not create duplicates.


In [0]:
# 1. Parameters

dbutils.widgets.text("start_date", "2026-09-01")
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("max_rows", "100")

start_date = dbutils.widgets.get("start_date").strip()
catalog_override = dbutils.widgets.get("catalog_name").strip()
max_rows_raw = dbutils.widgets.get("max_rows").strip()

if not start_date:
    raise ValueError("start_date is empty. Pass a value such as 2026-09-01.")

try:
    max_rows = int(max_rows_raw)
except ValueError:
    raise ValueError("max_rows must be an integer. Use 100 for testing or 0 for no limit.")

print("start_date =", start_date)
print("max_rows   =", max_rows)


In [0]:
# 2. Resolve Unity Catalog paths

if catalog_override:
    CATALOG = catalog_override
else:
    CATALOG = spark.sql(
        "SELECT current_catalog() AS catalog"
    ).first()["catalog"]

SCHEMA = "earthquake"

SILVER_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/silver"
GOLD_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/gold"

SILVER_INPUT_PATH = (
    f"{SILVER_ROOT}/earthquake_events_silver/"
    f"run_date={start_date}"
)

GOLD_OUTPUT_PATH = (
    f"{GOLD_ROOT}/earthquake_events_gold/"
    f"run_date={start_date}"
)

print("Catalog     :", CATALOG)
print("Silver input:", SILVER_INPUT_PATH)
print("Gold output :", GOLD_OUTPUT_PATH)


In [0]:
# 3. Read the Silver partition

from pyspark.sql.functions import col, when, current_timestamp, lit
import os

if not os.path.exists(SILVER_INPUT_PATH):
    raise FileNotFoundError(
        f"Silver data was not found at:\n{SILVER_INPUT_PATH}\n"
        "Run the fixed Silver notebook for this start_date first."
    )

df = spark.read.parquet(SILVER_INPUT_PATH)

required_columns = {
    "id",
    "longitude",
    "latitude",
    "elevation",
    "sig",
    "mag",
    "time"
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        "Silver data is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

print("Silver rows:", df.count())
df.printSchema()

display(df.limit(20))


## Install reverse geocoder

The tutorial Gold notebook uses `reverse_geocoder` to derive a country code from latitude and longitude.

For a learning project this is fine, but it can be slow on large datasets. The original tutorial also
limits the dataset during testing for that reason.


In [0]:
%pip install -q reverse_geocoder


In [0]:
# 4. Load reverse_geocoder

import reverse_geocoder as rg

print("reverse_geocoder loaded successfully")


In [0]:
# 5. Optional testing limit

# Keep the tutorial's testing behavior by default.
# Set max_rows = 0 in the widget if you want to process the entire date partition.

if max_rows > 0:
    df = df.limit(max_rows)

print("Rows to enrich:", df.count())


In [0]:
# 6. Country-code enrichment

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def get_country_code(lat, lon):
    if lat is None or lon is None:
        return None

    try:
        coordinates = (float(lat), float(lon))
        result = rg.search(coordinates, mode=1)

        if not result:
            return None

        return result[0].get("cc")

    except Exception:
        return None

get_country_code_udf = udf(
    get_country_code,
    StringType()
)

df_with_location = df.withColumn(
    "country_code",
    get_country_code_udf(
        col("latitude"),
        col("longitude")
    )
)


In [0]:
# 7. Significance classification

# Same thresholds as the tutorial:
# < 100      -> Low
# 100 to 499 -> Moderate
# >= 500     -> High

gold_df = (
    df_with_location

    .withColumn(
        "sig_class",
        when(
            col("sig") < 100,
            "Low"
        )
        .when(
            (col("sig") >= 100) & (col("sig") < 500),
            "Moderate"
        )
        .otherwise("High")
    )

    .withColumn(
        "gold_run_date",
        lit(start_date)
    )

    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print("=== GOLD SCHEMA ===")
gold_df.printSchema()

display(gold_df.limit(20))


In [0]:
# 8. Basic Gold quality checks

gold_df = (
    gold_df
    .filter(col("id").isNotNull())
    .dropDuplicates(["id"])
)

print("Gold row count:", gold_df.count())

display(
    gold_df
    .orderBy(col("time").desc())
    .limit(20)
)


In [0]:
# 9. Save Gold as Parquet

# Overwrite only this date partition.
# This makes rerunning the same date idempotent.

(
    gold_df
    .write
    .mode("overwrite")
    .parquet(GOLD_OUTPUT_PATH)
)

print("Gold successfully written to:")
print(GOLD_OUTPUT_PATH)


In [0]:
# 10. Verify Gold output

gold_check = spark.read.parquet(
    GOLD_OUTPUT_PATH
)

print("Verified Gold rows:", gold_check.count())
gold_check.printSchema()

display(gold_check.limit(20))


In [0]:
# 11. Inspect written files

for file_info in dbutils.fs.ls(GOLD_OUTPUT_PATH):
    print(file_info.path)


In [0]:
# 12. Parent Gold path for downstream analytics

GOLD_PARENT_PATH = (
    f"{GOLD_ROOT}/earthquake_events_gold/"
)

print("Gold parent path:", GOLD_PARENT_PATH)

# Optional for orchestration:
# dbutils.notebook.exit(GOLD_PARENT_PATH)
